# 05 — Model comparison and selection

**Goal:** check the model that `select_and_register` picked, before promoting it to `@champion`.
The pipeline's rule is fixed (`params.yaml: selection`): the best **val PR-AUC** among models
within the deployment limits (≤300 MB, p95 ≤300 ms per posting on one CPU thread). This notebook
asks how much that ranking can be trusted, whether the winner is servable, and whether its
calibrated probabilities mean what they say.

**Protocol** (same functions as the pipeline, via `src/fjd`):
- Every model is scored at its own val-tuned threshold, from the artifacts `dvc repro` produced.
- Ranking uncertainty: a **grouped paired bootstrap** (whole `group_id` groups resampled, every
  model scored on the same resamples), because val and test each hold only ~100 fraud postings and
  postings from one template are not independent.
- The test split is reported, never used to choose.

In [ ]:
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from mlflow import MlflowClient
from sklearn.calibration import calibration_curve
from sklearn.metrics import brier_score_loss

from fjd.data.io import REPORTS, read_split
from fjd.data.schema import TARGET
from fjd.features.build import INPUT_COLUMNS
from fjd.models.compare import group_bootstrap_pr_auc, paired_summary
from fjd.models.metrics import fraud_scores, pr_auc
from fjd.models.registry import Calibration, load_classifier
from fjd.models.select import check_deployability, model_path, ranking, threshold
from fjd.utils.config import load_params
from fjd.utils.plotting import BASELINE, INK_SECONDARY, SERIES, set_style
from fjd.utils.seed import set_seed
from fjd.utils.tracking import setup_mlflow

set_style()
params = load_params()
SEED, LIMITS = params["seed"], params["selection"]
METRIC = LIMITS["metric"]
N_RESAMPLES = 2000
set_seed(SEED)

metrics = json.loads((REPORTS / "metrics.json").read_text(encoding="utf-8"))
selection = json.loads((REPORTS / "selection.json").read_text(encoding="utf-8"))
splits = {"val": read_split("val"), "test": read_split("test")}
MODELS = ranking(metrics, METRIC)
SELECTED = selection["model"]
print(f"ranking by val {METRIC}: {MODELS}")
print(f"selected by the pipeline: {SELECTED}")

## 1. Leaderboard

From `reports/metrics.json` (the `evaluate` stage), best val PR-AUC first.

In [ ]:
board = pd.DataFrame(
    {
        name: {
            (split, k): metrics[name][split][k]
            for split in splits
            for k in ("pr_auc", "roc_auc", "f1", "precision", "recall")
        }
        for name in MODELS
    }
).T
board.style.format("{:.3f}")

## 2. How sure is the ranking?

Score every model on val and test once (all threads), check the scores reproduce the pipeline's
PR-AUC, then bootstrap.

In [ ]:
scores = {}
for name in MODELS:
    clf = load_classifier(name, model_path(name), threads=None)
    scores[name] = {s: fraud_scores(clf, df[INPUT_COLUMNS]) for s, df in splits.items()}
    for s, df in splits.items():
        # Classical models match exactly; the transformer within its repro tolerance.
        assert abs(pr_auc(df[TARGET], scores[name][s]) - metrics[name][s]["pr_auc"]) < 1e-3

In [ ]:
boot, summary = {}, {}
for s, df in splits.items():
    by_model = {n: scores[n][s] for n in MODELS}
    boot[s] = group_bootstrap_pr_auc(df[TARGET], by_model, df["group_id"], N_RESAMPLES, SEED)
    summary[s] = paired_summary(boot[s], SELECTED)


def show(table: pd.DataFrame) -> pd.io.formats.style.Styler:
    return (
        table.style.format("{:+.3f}")
        .format("{:.3f}", subset=["pr_auc_mean"])
        .format("{:.0%}", subset=["p_beats_reference"])
    )


print(f"val: PR-AUC difference from {SELECTED} over {N_RESAMPLES} grouped resamples")
show(summary["val"])

In [ ]:
print(f"test (reported only): PR-AUC difference from {SELECTED}")
show(summary["test"])

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
order = summary["val"].index  # same model order in both panels
y = np.arange(len(order))[::-1]
colors = [SERIES[1] if n == SELECTED else SERIES[0] for n in order]
for ax, (s, b) in zip(axes, boot.items(), strict=True):
    lo, mid, hi = (b[order].quantile(q) for q in (0.025, 0.5, 0.975))
    ax.hlines(y, lo, hi, color=colors, lw=2)
    ax.scatter(mid, y, color=colors, zorder=3)
    ax.scatter([metrics[n][s]["pr_auc"] for n in order], y, marker="|", s=120, color=INK_SECONDARY)
    ax.set_yticks(y, order)
    ax.set_title(f"{s}: PR-AUC, 95% grouped-bootstrap interval", fontsize=10)
    ax.set_xlabel("PR-AUC (dot = bootstrap median, tick = full split)")
plt.tight_layout()
plt.show()

## 3. Deployability

Measured here as the registry serves: one posting at a time on one thread
(`fjd.models.select.check_deployability`). Latency depends on the machine; Render's free tier is
slower than a laptop, so a model close to the limit here is a risk there.

In [ ]:
val = splits["val"]
sample = val.sample(min(LIMITS["latency_sample"], len(val)), random_state=SEED)
deploy = pd.DataFrame({name: check_deployability(name, sample, LIMITS) for name in MODELS}).T
deploy

## 4. Calibration

The registered pyfunc maps scores to P(fraud) with a Platt sigmoid fitted on val
(`fjd.models.registry.Calibration`). Raw `predict_proba` from class-weighted models is inflated,
and LinearSVC has only margins. Reliability on **test** (not used for fitting), 10 quantile bins.

In [ ]:
y_val, y_test = splits["val"][TARGET], splits["test"][TARGET]
calibrations = {name: Calibration.fit(y_val, scores[name]["val"]) for name in MODELS}
curves = {f"{n} (Platt)": calibrations[n](scores[n]["test"]) for n in MODELS[:3]}
if "logreg" in MODELS:  # class_weight="balanced" probabilities, before calibration
    curves["logreg (raw predict_proba)"] = scores["logreg"]["test"]

fig, ax = plt.subplots(figsize=(5.5, 4.5))
ax.plot([0, 1], [0, 1], color=BASELINE, lw=1, ls="--")
rows = {}
for color, (label, p) in zip(SERIES, curves.items(), strict=False):
    observed, predicted = calibration_curve(y_test, p, n_bins=10, strategy="quantile")
    ax.plot(predicted, observed, marker="o", ms=4, color=color, label=label)
    rows[label] = {"test brier": brier_score_loss(y_test, p), "mean P(fraud)": p.mean()}
ax.set(xlabel="mean predicted P(fraud)", ylabel="observed fraud rate", xlim=(0, 1), ylim=(0, 1))
ax.set_title("Test reliability", fontsize=10)
ax.legend(loc="upper left", fontsize=8)
plt.tight_layout()
plt.show()

constant = np.full(len(y_test), y_val.mean())
rows["constant: val fraud rate"] = {
    "test brier": brier_score_loss(y_test, constant),
    "mean P(fraud)": y_val.mean(),
}
print(f"test fraud rate {y_test.mean():.4f}")
pd.DataFrame(rows).T.style.format("{:.4f}")

In [ ]:
t = threshold(SELECTED)
t_prob = calibrations[SELECTED](np.array([t]))[0]
print(f"{SELECTED}: decision threshold {t:.4f} on the raw score = P(fraud) {t_prob:.3f}")

## 5. Registry

Versions of the registered model in the current MLflow store (DagsHub if `MLFLOW_TRACKING_URI`
is set, else the local `mlflow.db`). `dvc repro` registers a new `@challenger`; promote it with
`uv run poe promote`.

In [ ]:
print("MLflow:", setup_mlflow())
client = MlflowClient()
registered = params["mlflow"]["registered_model"]
# search_model_versions leaves `aliases` empty; get_model_version fills it in.
versions = [
    client.get_model_version(registered, v.version)
    for v in client.search_model_versions(f"name = '{registered}'")
]
pd.DataFrame(
    [
        {
            "version": int(v.version),
            "model": v.tags.get("model_name"),
            "aliases": ", ".join(v.aliases),
            "created": pd.to_datetime(v.creation_timestamp, unit="ms"),
        }
        for v in versions
    ],
    columns=["version", "model", "aliases", "created"],
).sort_values("version")

## 6. Takeaways

Numbers from the run on 2026-10-26 (seed 42, 2000 grouped resamples; latency on a Windows laptop).

1. **The pipeline's pick, LinearSVM, is a statistical tie with logistic regression.** Their val
   PR-AUC differs by 0.002 (95% CI −0.014 to +0.009); the two linear models rank postings almost
   identically. Either is a sound champion, and both suit the planned coefficient × TF-IDF
   explanations. Keeping the fixed rule (best val PR-AUC) avoids choosing by eye.
2. **Beyond the linear pair, val cannot separate the families.** Intervals on the difference
   are about ±0.2, because ~100 fraud postings fall into far fewer independent groups. On test the
   boosters and the random forest lead the SVM by 0.002–0.010, also well inside the noise. Only
   Naive Bayes is clearly worse (test CI entirely below zero). Read the differences, not the
   bootstrap means: resampling whole groups inflates absolute PR-AUC.
3. **The transformer is not servable on a laptop CPU, let alone Render.** Its p95 is ~590 ms here
   against 254 ms on Colab, and its val PR-AUC is no better than the linear models'. Every
   classical model fits the limits (≤18 MB, p95 ≤ 280 ms; the linear models ~60 ms).
4. **Calibration helps, but the probabilities run low on new groups.** Platt scaling beats the
   constant base-rate forecast by far (test Brier 0.018 vs 0.043), and the threshold maps to
   P(fraud) ≈ 0.34. On test, though, the mean calibrated P(fraud) is 0.030 against a fraud rate of
   0.045: test groups get lower SVM margins than val groups. Show the probability as a guide in the
   UI; the label comes from the tuned threshold, which is what the metrics describe.
5. **Next:** promote with `uv run poe promote` once the challenger is registered on DagsHub, then
   build the API on `@champion` (phase 6).